# Upload trained weights to the Hugging Face Hub

Pushes your two trained checkpoints (on Google Drive) to public HF model repos so `texture_frames_de.FrameParser()` can download them. Run once, after training.

**Publishes model weights derived from SALSA/TIGER (academic/non-commercial).** Ensure this is consistent with the corpus licence before running (see the repo README's licence section).

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!pip install -q -U "huggingface_hub>=0.20"

In [ ]:
from huggingface_hub import login
login()   # paste a HF token with *write* access (hf.co/settings/tokens)

In [ ]:
USER   = "texturejc"
FRAME_REPO = f"{USER}/texture-frames-de-frame"
ARGS_REPO  = f"{USER}/texture-frames-de-args"
FRAME_DIR  = "/content/drive/MyDrive/Texture_Frames/models/frame2_de"
ARGS_DIR   = "/content/drive/MyDrive/Texture_Frames/models/args2_de"

import os
for d, files in [(FRAME_DIR, ["frame2_model.pt","frame2id.json"]),
                 (ARGS_DIR,  ["args2_model.pt","role2id.json"])]:
    for f in files:
        assert os.path.exists(os.path.join(d, f)), f"missing {f} in {d}"
print("checkpoints present.")

## Push both repos

Uploads the model `.pt`, its id-map JSON, and the tokenizer files. Public by default.

In [ ]:
from huggingface_hub import HfApi, create_repo
api = HfApi()

for repo, folder in [(FRAME_REPO, FRAME_DIR), (ARGS_REPO, ARGS_DIR)]:
    create_repo(repo, repo_type="model", exist_ok=True, private=False)
    api.upload_folder(repo_id=repo, folder_path=folder, repo_type="model")
    print("pushed ->", f"https://huggingface.co/{repo}")

## Verify it loads

Round-trips through the package's loader (downloads from the Hub you just pushed).

In [ ]:
!pip install -q "transformers==4.57.6" sentencepiece simplemma
import sys; sys.path.insert(0, "/content/texture-frames-de/src")  # if repo cloned here
from texture_frames_de import FrameParser
parser = FrameParser(frame_repo=FRAME_REPO, args_repo=ARGS_REPO)
print(parser.parse("Die Polizei verhaftete den Verdächtigen ."))